In [1]:
# this script gathers HTML files using the singlefile docker through subprocess. 
# first, define the language to be processed
LANGUAGE_TO_PROCESS = "es"
import subprocess
import os
import json

In [2]:
# read the JSON file of the language to be processed
urls_to_download = []
json_data = []

with open(f"mhtml_results_{LANGUAGE_TO_PROCESS}.json", "r") as f:
    json_data = json.load(f)

if LANGUAGE_TO_PROCESS == "en" or LANGUAGE_TO_PROCESS == "zh": 
    for item in json_data: 
        timestamp = item['timestamp']
        original_url = item['url']
        item_url = "https://web.archive.org/web/{timestamp}if_/{original_url}".format(timestamp=timestamp, original_url=original_url)
        urls_to_download.append({
            "banner_md5": item["md5"],
            "url": item_url}
            )
elif LANGUAGE_TO_PROCESS == "es" or "ja":
    for item in json_data: 
        mhtml_file = item['mhtml_file']
        # open the mhtml file and read the Snapshot-Content-Location
        snapshot_content_location = ""
        with open(mhtml_file, "r") as f:
            content = f.read()
            snapshot_content_location = content.split("Snapshot-Content-Location: ")[1].split("\n")[0]
        # only append if banner_md5 is not already in the list
        if not any(d['banner_md5'] == item["banner_md5"] for d in urls_to_download):
            urls_to_download.append({
                'banner_md5': item["banner_md5"],
                'url': snapshot_content_location}
                )
else:
    print("Language not supported.")
    exit()


In [3]:
urls_to_download[10]

{'banner_md5': 'b9b0e4bfc347a16426f456937ce74ffb',
 'url': 'https://web.archive.org/web/20010620141853if_/http://www.postalycual.com/'}

In [4]:
# podman command: podman run singlefile "{url}" > {banner_md5}.html

# check how many files are there already downloaded in the directory
download_directory = f"html_{LANGUAGE_TO_PROCESS}"
# get a list of .html files in the directory
html_files = [f for f in os.listdir(download_directory) if f.endswith('.html')]
downloaded_md5s = [f.split(".html")[0] for f in html_files]

md5s_to_download = [item['banner_md5'] for item in urls_to_download if item['banner_md5'] not in downloaded_md5s]
# shuffle the list
import random
random.shuffle(md5s_to_download)

# download the files
for item in urls_to_download:
    if item['banner_md5'] in md5s_to_download:
        url = item['url']
        banner_md5 = item['banner_md5']
        print(f"Downloading {url}")
        command = f"podman run singlefile {url} > {download_directory}/{banner_md5}.html"
        subprocess.run(command, shell=True)
        print(f"Downloaded {url}")


Downloaded https://web.archive.org/web/20010619032748if_/http://www.postalycual.com/


Unreachable URL: https://web.archive.org/web/20010620140415if_/http://www.postalycual.com/ [2024-09-21T06:43:25.726Z] URL: https://web.archive.org/web/20010620140415if_/http://www.postalycual.com/



Downloaded https://web.archive.org/web/20010620140415if_/http://www.postalycual.com/


KeyboardInterrupt: 